# CityBrain v9 — Clean Feature Separation + Neighbourhood Embedding

**Changes from v8:**
1. STRICT feature separation: Road = physical ONLY (8d), Tabular = context ONLY (34d)
   - ZERO feature overlap between branches
   - Cross-attention now has genuinely different information to fuse
2. Neighbourhood entity embedding (22d one-hot -> 4d learned embedding)
3. Multi-scale spatial lag (3-NN + 7-NN + 15-NN)
4. Complaint density (complaints / road length)
5. CatBoost added (native categorical support for neighbourhood)
6. 3-model ensemble (Fusion + XGBoost + CatBoost)

## 0. Setup

In [17]:
import os, ast, warnings
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.optimize import differential_evolution
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
warnings.filterwarnings('ignore')

# --- Colab ---
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/AI-FinalProject/data'

SNAP_RADIUS_M = 150
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

LABEL_MAP = {
    'VERY GOOD': 0,
    'GOOD': 1, 'FAIR': 1,
    'POOR': 2, 'VERY POOR': 2,
}
NUM_CLASSES = 3
CLASS_NAMES = ['Low', 'Medium', 'High']

def safe_load(filename, **kwargs):
    path = os.path.join(DATA_DIR, filename) if not os.path.isabs(filename) else filename
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        first = f.readline()
    sep = ';' if first.count(';') > first.count(',') else ','
    return pd.read_csv(path, sep=sep, on_bad_lines='skip', **kwargs)

print(f'Label mapping: {LABEL_MAP}')
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Label mapping: {'VERY GOOD': 0, 'GOOD': 1, 'FAIR': 1, 'POOR': 2, 'VERY POOR': 2}
Classes (3): ['Low', 'Medium', 'High']


## 1. Load & Merge Data

In [18]:
enriched_path = '/content/drive/MyDrive/AI-FinalProject/data/pavement_enriched.csv'
if os.path.exists(enriched_path):
    df = pd.read_csv(enriched_path)
    print(f'Loaded pavement_enriched.csv: {len(df):,} rows')
else:
    df_local = safe_load('pavement_condition.csv'); df_local['road_type'] = 'local'
    df_major = safe_load('pavement_condition_major_road_network.csv'); df_major['road_type'] = 'major'
    df = pd.concat([df_local, df_major], ignore_index=True)
    coords = df['geo_point_2d'].str.split(',', expand=True).astype(float)
    df['lat'], df['lon'] = coords[0], coords[1]

df = df[df['PCI Rating'].isin(LABEL_MAP)].copy()
df['risk_label'] = df['PCI Rating'].map(LABEL_MAP)
df['source'] = (df['road_type'] == 'major').astype(int) if 'road_type' in df.columns else 0
print(f'Total samples: {len(df):,} (ALL kept)')
for v, name in enumerate(CLASS_NAMES):
    cnt = (df['risk_label']==v).sum()
    print(f'  {v} ({name}): {cnt:,} ({cnt/len(df)*100:.1f}%)')

pave_coords = np.column_stack([df['lat'].values, df['lon'].values])
LAT_M, LON_M = 111_000, 73_000

# --- streetuse ---
df_st = pd.read_csv('/content/drive/MyDrive/AI-FinalProject/data/public_streets.csv',
                     quotechar='"', on_bad_lines='skip')
st_geo = df_st['geo_point_2d'].dropna().apply(ast.literal_eval)
st_coords = np.array([[d['lat'], d['lon']] for d in st_geo])
_, idx = cKDTree(st_coords).query(pave_coords)
df['streetuse'] = df_st.loc[st_geo.index, 'streetuse'].values[idx]
STREETUSE_WEIGHT = {'Residential': 0.3, 'Collector': 0.6, 'Arterial': 1.0}
df['streetuse_weight'] = df['streetuse'].map(STREETUSE_WEIGHT).fillna(0.5)

# --- ROW width ---
df_row = safe_load('right_of_way_widths.csv')
geo_col = [c for c in df_row.columns if 'geo_point' in c.lower()][0]
width_col = [c for c in df_row.columns if 'width' in c.lower()][0]
row_geo = df_row[geo_col].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
row_coords = np.array([[d['lat'], d['lon']] for d in row_geo if isinstance(d, dict)])
row_widths = pd.to_numeric(df_row.loc[row_geo.index[:len(row_coords)], width_col], errors='coerce').fillna(0).values
_, idx = cKDTree(row_coords).query(pave_coords)
df['ROW_width'] = row_widths[idx]

# --- repair_count ---
df['repair_count'] = 0
repair_path = '/content/drive/MyDrive/AI-FinalProject/data/city_project_package_street.csv'
if os.path.exists(repair_path):
    df_repair = safe_load('city_project_package_street.csv')
    geo_cols = [c for c in df_repair.columns if 'geo_point' in c.lower()]
    if geo_cols:
        try:
            rg = df_repair[geo_cols[0]].dropna()
            parsed = rg.apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
            rc = np.array([[d['lat'], d['lon']] for d in parsed if isinstance(d, dict)])
            counts = cKDTree(rc * [LAT_M, LON_M]).query_ball_point(pave_coords * [LAT_M, LON_M], r=100)
            df['repair_count'] = [len(c) for c in counts]
        except: pass

# --- segment_density ---
tree_all = cKDTree(pave_coords)
neighbours = tree_all.query_ball_point(pave_coords, r=0.003)
df['segment_density'] = [len(n) - 1 for n in neighbours]

# --- elevation & slope ---
if 'elevation_m' not in df.columns:
    df['elevation_m'] = 0.0; df['slope_pct'] = 0.0
df['elevation_m'] = df['elevation_m'].fillna(df['elevation_m'].median())
df['slope_pct'] = df['slope_pct'].fillna(df['slope_pct'].median())

# --- neighbourhood ---
df['neighbourhood'] = df.get('neighbourhood', pd.Series('Unknown', index=df.index)).fillna('Unknown')

# --- traffic counter distance ---
df_traffic = safe_load('directional_traffic_count_locations.csv')
traf_geo_col = [c for c in df_traffic.columns if 'geo_point' in c.lower()][0]
traf_geo = df_traffic[traf_geo_col].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
traf_coords = np.array([[d['lat'], d['lon']] for d in traf_geo if isinstance(d, dict)])
traf_dists, _ = cKDTree(traf_coords * [LAT_M, LON_M]).query(pave_coords * [LAT_M, LON_M])
df['dist_to_traffic_count'] = traf_dists

print(f'Static features ready.')

Loaded pavement_enriched.csv: 13,764 rows
Total samples: 13,764 (ALL kept)
  0 (Low): 3,247 (23.6%)
  1 (Medium): 5,894 (42.8%)
  2 (High): 4,623 (33.6%)
Static features ready.


## 2. Train / Val / Test Split

In [19]:
y = df['risk_label'].values
indices = np.arange(len(df))
idx_train, idx_temp = train_test_split(indices, test_size=0.30, random_state=42, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42, stratify=y[idx_temp])
print(f'Train: {len(idx_train):,}  Val: {len(idx_val):,}  Test: {len(idx_test):,}')

Train: 9,634  Val: 2,065  Test: 2,065


## 3. ★ v9: Multi-Scale Spatial Lag (3/7/15-NN) ★

In [20]:
print('\n--- Computing Multi-Scale Spatial Lag Features ---')
train_coords_m = pave_coords[idx_train] * [LAT_M, LON_M]
all_coords_m   = pave_coords * [LAT_M, LON_M]
train_tree_sl  = cKDTree(train_coords_m)
train_labels   = y[idx_train]

nn_k_7 = None
sl_weights_n_7 = None

for K in [3, 7, 15]:
    dists_k, nn_k = train_tree_sl.query(all_coords_m, k=K + 1)
    self_match = dists_k[:, 0] < 0.01
    dists_k = np.where(self_match[:, None], dists_k[:, 1:K+1], dists_k[:, :K])
    nn_k    = np.where(self_match[:, None], nn_k[:, 1:K+1],    nn_k[:, :K])

    sl_weights   = 1.0 / (dists_k + 1e-6)
    sl_weights_n = sl_weights / sl_weights.sum(axis=1, keepdims=True)
    neighbour_labs = train_labels[nn_k]

    df[f'spatial_lag_risk_{K}']  = (sl_weights_n * neighbour_labs).sum(axis=1)
    df[f'spatial_lag_high_{K}']  = (sl_weights_n * (neighbour_labs == 2)).sum(axis=1)
    if K == 7:
        df[f'spatial_lag_med_{K}'] = (sl_weights_n * (neighbour_labs == 1)).sum(axis=1)
        nn_k_7, sl_weights_n_7 = nn_k.copy(), sl_weights_n.copy()

    if K == 3:
        print(f'  Self-matches removed: {self_match.sum()} / {len(self_match)}')
    m = df[f'spatial_lag_risk_{K}']
    print(f'  {K}-NN: risk mean={m.mean():.3f} std={m.std():.3f}')


--- Computing Multi-Scale Spatial Lag Features ---
  Self-matches removed: 9634 / 13764
  3-NN: risk mean=1.096 std=0.550
  7-NN: risk mean=1.099 std=0.453
  15-NN: risk mean=1.099 std=0.383


## 4. Neighbourhood Target Encoding (train-only)

In [21]:
print('\n--- Computing Neighbourhood Target Encoding ---')
train_df = df.iloc[idx_train]
for cls, name in [(0, 'neigh_low_pct'), (1, 'neigh_med_pct'), (2, 'neigh_high_pct')]:
    d = train_df.groupby('neighbourhood')['risk_label'].apply(
        lambda x: (x == cls).mean()).to_dict()
    global_rate = (train_df['risk_label'] == cls).mean()
    df[name] = df['neighbourhood'].map(d).fillna(global_rate)


--- Computing Neighbourhood Target Encoding ---


## 5. Weather & 311 → Static Features

In [22]:
print('\n--- Computing Weather & 311 Static Features ---')

# --- Weather ---
df_w = safe_load('weather_vancouver.csv')
date_col = [c for c in df_w.columns if 'date' in c.lower() or 'time' in c.lower()][0]
df_w['date'] = pd.to_datetime(df_w[date_col], errors='coerce')
df_w = df_w.dropna(subset=['date'])
col_map = {}
for c in df_w.columns:
    cl = c.lower().strip()
    if 'max temp' in cl and 'flag' not in cl: col_map[c] = 'max_temp'
    elif 'min temp' in cl and 'flag' not in cl: col_map[c] = 'min_temp'
    elif 'total precip' in cl and 'flag' not in cl: col_map[c] = 'total_precip'
    elif 'total snow' in cl and 'flag' not in cl: col_map[c] = 'total_snow'
df_w = df_w.rename(columns=col_map)
for c in ['max_temp','min_temp','total_precip','total_snow']:
    if c in df_w.columns: df_w[c] = pd.to_numeric(df_w[c], errors='coerce').fillna(0)
df_w['freeze_thaw'] = ((df_w['max_temp'] > 0) & (df_w['min_temp'] < 0)).astype(int)
df_w['extreme'] = (df_w['total_precip'] > df_w['total_precip'].quantile(0.95)).astype(int)
df_w['year'] = df_w['date'].dt.year

annual_wx = df_w.groupby('year').agg(
    annual_precip=('total_precip', 'sum'),
    annual_snow=('total_snow', 'sum'),
    annual_freeze=('freeze_thaw', 'sum'),
    annual_extreme=('extreme', 'sum'),
    annual_temp_range=('max_temp', lambda x: x.mean() - df_w.loc[x.index, 'min_temp'].mean()),
).to_dict('index')
print(f'Weather years available: {sorted(annual_wx.keys())}')

years = df['Year'].values if 'Year' in df.columns else np.full(len(df), 2020)
slopes = df['slope_pct'].values
su_weights_arr = df['streetuse_weight'].values

for feat in ['annual_precip', 'annual_snow', 'annual_freeze', 'annual_extreme', 'annual_temp_range']:
    df[feat] = [annual_wx.get(yr, {}).get(feat, 0) for yr in years]

df['prev_yr_precip'] = [annual_wx.get(yr - 1, {}).get('annual_precip', 0) for yr in years]
df['prev_yr_freeze'] = [annual_wx.get(yr - 1, {}).get('annual_freeze', 0) for yr in years]

# Weather × road interactions
df['precip_x_slope']     = df['annual_precip'] * slopes
df['freeze_x_street']    = df['annual_freeze'] * su_weights_arr
df['snow_x_slope']       = df['annual_snow'] * slopes
df['temprange_x_street'] = df['annual_temp_range'] * su_weights_arr

print(f'  Weather: 7 aggregates + 4 interactions = 11')

# --- 311 ---
print('\nLoading 311...')
df_311 = safe_load('311_service_requests_2009_2021.csv', low_memory=False)
id_cols = [c for c in df_311.columns if 'case' in c.lower() or 'id' in c.lower()]
if id_cols: df_311 = df_311.drop_duplicates(subset=id_cols[0])
PAVEMENT_TYPES = [
    'Pothole Case', 'Street Repair Case', 'Sidewalk Repair Case',
    'Pavement Marking Maintenance Case', 'Pavement Markings Case',
    'Street Surface Water Flooding Case', 'Street Construction Concern Case',
    'Street Cleaning and Debris Pick Up Case', 'Curb Ramp Request Case',
]
type_col = 'Service request type'
df_311 = df_311[df_311[type_col].isin(PAVEMENT_TYPES)]
print(f'  Pavement-related: {len(df_311):,}')

date_cols = [c for c in df_311.columns if 'open' in c.lower() or 'timestamp' in c.lower() or 'created' in c.lower()]
if not date_cols: date_cols = [c for c in df_311.columns if 'date' in c.lower()]
df_311['date'] = pd.to_datetime(df_311[date_cols[0]], errors='coerce', utc=True)
lat_col = [c for c in df_311.columns if c.lower() in ['latitude','lat']]
lon_col = [c for c in df_311.columns if c.lower() in ['longitude','lon']]
if lat_col and lon_col:
    df_311['c_lat'] = pd.to_numeric(df_311[lat_col[0]], errors='coerce')
    df_311['c_lon'] = pd.to_numeric(df_311[lon_col[0]], errors='coerce')
else:
    geo_col = [c for c in df_311.columns if 'geo_local' in c.lower() or 'geo_point' in c.lower()]
    if geo_col:
        raw = df_311[geo_col[0]].dropna()
        if len(raw) > 0 and '{' in str(raw.iloc[0]):
            parsed = raw.apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else s)
            df_311.loc[raw.index, 'c_lat'] = [d.get('lat',np.nan) if isinstance(d,dict) else np.nan for d in parsed]
            df_311.loc[raw.index, 'c_lon'] = [d.get('lon',np.nan) if isinstance(d,dict) else np.nan for d in parsed]

df_311 = df_311.dropna(subset=['c_lat','c_lon','date'])
df_311['year'], df_311['month'] = df_311['date'].dt.year, df_311['date'].dt.month
survey_years = df['Year'].unique() if 'Year' in df.columns else [2020, 2021]
df_311 = df_311[df_311['year'].isin(survey_years)]
print(f'  Survey-year 311: {len(df_311):,}')

c311_coords = df_311[['c_lat','c_lon']].values
pave_tree_m = cKDTree(pave_coords * [LAT_M, LON_M])
dists_311, snap_idx = pave_tree_m.query(c311_coords * [LAT_M, LON_M])
mask = dists_311 < SNAP_RADIUS_M
df_311 = df_311[mask].copy()
df_311['seg_idx'] = snap_idx[mask]
df_311['is_pothole'] = df_311[type_col].str.contains('Pothole', case=False).astype(int)
print(f'  Snapped within {SNAP_RADIUS_M}m: {len(df_311):,}')

N = len(df)
complaint_total = np.zeros(N)
pothole_total = np.zeros(N)
complaint_recent = np.zeros(N)

for i in range(N):
    yr = years[i]
    seg_c = df_311[(df_311['seg_idx'] == i) & (df_311['year'] == yr)]
    complaint_total[i] = len(seg_c)
    pothole_total[i] = seg_c['is_pothole'].sum()
    complaint_recent[i] = len(seg_c[seg_c['month'] >= 10])

df['complaint_total'] = complaint_total
df['pothole_total'] = pothole_total
df['complaint_recent_3m'] = complaint_recent
df['complaint_accel'] = complaint_recent / np.maximum(complaint_total, 1)

# ★ v9: complaint density (normalize by road length) ★
lengths = df['length_(m)'].fillna(df['length_(m)'].median()).values
df['complaint_density'] = complaint_total / np.maximum(lengths, 1.0)

# Spatial lag of complaint counts (7-NN)
train_complaints = complaint_total[idx_train]
neighbour_compl = train_complaints[nn_k_7]
df['spatial_lag_complaints'] = (sl_weights_n_7 * neighbour_compl).sum(axis=1)

print(f'  Complaint features: total, pothole, recent_3m, accel, density, sl_complaints')
print(f'  complaint_total nonzero: {(complaint_total > 0).sum():,} / {N:,}')


--- Computing Weather & 311 Static Features ---
Weather years available: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
  Weather: 7 aggregates + 4 interactions = 11

Loading 311...
  Pavement-related: 135,647
  Survey-year 311: 27,187
  Snapped within 150m: 27,024
  Complaint features: total, pothole, recent_3m, accel, density, sl_complaints
  complaint_total nonzero: 6,204 / 13,764


## 6. ★ v9: Feature Construction (CLEAN SEPARATION) ★

In [23]:
print('\n--- Feature Construction (CLEAN SEPARATION) ---')

su_dummies = pd.get_dummies(df['streetuse'], prefix='su')
for c in ['su_Arterial','su_Collector','su_Residential']:
    if c not in su_dummies.columns: su_dummies[c] = 0

# ★ Road-MLP: 8d PURE PHYSICAL — what IS this road ★
X_road_raw = np.column_stack([
    lengths,
    su_dummies[['su_Arterial','su_Collector','su_Residential']].values,
    df['elevation_m'].values,
    df['slope_pct'].values,
    df['source'].values,
    df['ROW_width'].values,
]).astype(np.float32)

# ★ Tabular-MLP continuous: 30d — what's AROUND / HAPPENING TO this road ★
X_tab_cont_raw = np.column_stack([
    # --- Spatial context (10) ---
    df['repair_count'].values,
    df['segment_density'].values,
    df['dist_to_traffic_count'].values,
    df['spatial_lag_risk_3'].values,
    df['spatial_lag_high_3'].values,
    df['spatial_lag_risk_7'].values,
    df['spatial_lag_high_7'].values,
    df['spatial_lag_med_7'].values,
    df['spatial_lag_risk_15'].values,
    df['spatial_lag_high_15'].values,
    # --- Neighbourhood encoding (3) ---
    df['neigh_low_pct'].values,
    df['neigh_med_pct'].values,
    df['neigh_high_pct'].values,
    # --- Complaints (6) ---
    df['complaint_total'].values,
    df['pothole_total'].values,
    df['complaint_recent_3m'].values,
    df['complaint_accel'].values,
    df['complaint_density'].values,
    df['spatial_lag_complaints'].values,
    # --- Weather (7) ---
    df['annual_precip'].values,
    df['annual_snow'].values,
    df['annual_freeze'].values,
    df['annual_extreme'].values,
    df['annual_temp_range'].values,
    df['prev_yr_precip'].values,
    df['prev_yr_freeze'].values,
    # --- Weather × road interactions (4) ---
    df['precip_x_slope'].values,
    df['freeze_x_street'].values,
    df['snow_x_slope'].values,
    df['temprange_x_street'].values,
]).astype(np.float32)

# ★ Neighbourhood → integer index for embedding ★
neigh_cat = df['neighbourhood'].astype('category')
X_tab_cat_raw = neigh_cat.cat.codes.values.astype(np.int64)
N_NEIGH = len(neigh_cat.cat.categories)
NEIGH_EMB_DIM = 4
TAB_CONT_DIM = X_tab_cont_raw.shape[1]

print(f'Road-MLP:       {X_road_raw.shape[1]}d (pure physical)')
print(f'Tabular-MLP:    {TAB_CONT_DIM}d continuous + {NEIGH_EMB_DIM}d neighbourhood embedding = {TAB_CONT_DIM + NEIGH_EMB_DIM}d')
print(f'Neighbourhoods: {N_NEIGH} unique → {NEIGH_EMB_DIM}d embedding (was {N_NEIGH}d one-hot)')
print(f'Feature overlap between branches: ZERO')


--- Feature Construction (CLEAN SEPARATION) ---
Road-MLP:       8d (pure physical)
Tabular-MLP:    30d continuous + 4d neighbourhood embedding = 34d
Neighbourhoods: 23 unique → 4d embedding (was 23d one-hot)
Feature overlap between branches: ZERO


## 7. Normalize + RandomOverSampler (neural nets only)

In [24]:
sc_road = StandardScaler().fit(X_road_raw[idx_train])
sc_tab  = StandardScaler().fit(X_tab_cont_raw[idx_train])

Xr_tr = sc_road.transform(X_road_raw[idx_train]).astype(np.float32)
Xr_v  = sc_road.transform(X_road_raw[idx_val]).astype(np.float32)
Xr_te = sc_road.transform(X_road_raw[idx_test]).astype(np.float32)

# Tabular: scale continuous, then append UNscaled neigh index as last column
Xbc_tr = sc_tab.transform(X_tab_cont_raw[idx_train]).astype(np.float32)
Xbc_v  = sc_tab.transform(X_tab_cont_raw[idx_val]).astype(np.float32)
Xbc_te = sc_tab.transform(X_tab_cont_raw[idx_test]).astype(np.float32)

Xb_tr = np.column_stack([Xbc_tr, X_tab_cat_raw[idx_train].astype(np.float32)])
Xb_v  = np.column_stack([Xbc_v,  X_tab_cat_raw[idx_val].astype(np.float32)])
Xb_te = np.column_stack([Xbc_te, X_tab_cat_raw[idx_test].astype(np.float32)])

y_tr, y_v, y_te = y[idx_train], y[idx_val], y[idx_test]

# Save originals for tree models
Xr_tr_orig, Xb_tr_orig, y_tr_orig = Xr_tr.copy(), Xb_tr.copy(), y_tr.copy()

# ROS for neural nets only
try:
    from imblearn.over_sampling import RandomOverSampler
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'])
    from imblearn.over_sampling import RandomOverSampler

print(f'\nBefore ROS: {np.bincount(y_tr)}')
flat_tr = np.hstack([Xr_tr, Xb_tr])
ros = RandomOverSampler(random_state=42)
_, _ = ros.fit_resample(flat_tr, y_tr)
res_idx = ros.sample_indices_
Xr_tr = Xr_tr[res_idx]
Xb_tr = Xb_tr[res_idx]
y_tr  = y_tr[res_idx]
print(f'After  ROS: {np.bincount(y_tr)}')

class_counts = np.bincount(y_tr, minlength=NUM_CLASSES)
cw = 1.0 / class_counts; cw = cw / cw.sum() * NUM_CLASSES
CW = torch.FloatTensor(cw).to(DEVICE)
print(f'Road: {Xr_tr.shape}, Tabular: {Xb_tr.shape} (last col = neigh index)')


Before ROS: [2273 4125 3236]
After  ROS: [4125 4125 4125]
Road: (12375, 8), Tabular: (12375, 31) (last col = neigh index)


## 8. Loss Functions

In [25]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha, self.gamma, self.label_smoothing = alpha, gamma, label_smoothing
    def forward(self, logits, targets):
        nc = logits.size(1)
        if self.label_smoothing > 0:
            with torch.no_grad():
                smooth = torch.full_like(logits, self.label_smoothing / (nc - 1))
                smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
            loss = -(smooth * torch.log_softmax(logits, dim=1)).sum(dim=1)
        else:
            loss = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.softmax(logits, dim=1).gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = (1 - p_t) ** self.gamma * loss
        if self.alpha is not None: loss = self.alpha[targets] * loss
        return loss.mean()

class OrdinalPenalty(nn.Module):
    def __init__(self, weight=0.3):
        super().__init__()
        self.weight = weight
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        classes = torch.arange(logits.size(1), device=logits.device, dtype=torch.float32)
        expected = (probs * classes.unsqueeze(0)).sum(dim=1)
        return self.weight * ((expected - targets.float()) ** 2).mean()

## 9. ★ v9: Branch Models (CLEAN — no overlap) ★

In [26]:
class RoadMLP(nn.Module):
    """Pure physical road attributes (8d)"""
    def __init__(self, dim, emb=16, hidden=64, drop=0.4):
        super().__init__()
        self.emb_dim = emb
        self.enc = nn.Sequential(
            nn.Linear(dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, emb), nn.ReLU())
        self.head = nn.Linear(emb, NUM_CLASSES)
    def get_embedding(self, x): return self.enc(x)
    def forward(self, x): return self.head(self.enc(x))

class TabularMLP(nn.Module):
    """Context features (30d continuous) + neighbourhood embedding (4d)
    Input tensor: (B, cont_dim+1) where last column = neighbourhood index"""
    def __init__(self, cont_dim, n_neigh, neigh_emb_dim=4, emb=16, hidden=128, drop=0.4):
        super().__init__()
        self.emb_dim = emb
        self.cont_dim = cont_dim
        self.neigh_emb = nn.Embedding(n_neigh, neigh_emb_dim)
        total = cont_dim + neigh_emb_dim
        self.enc = nn.Sequential(
            nn.Linear(total, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, emb), nn.ReLU())
        self.head = nn.Linear(emb, NUM_CLASSES)

    def _unpack(self, x):
        x_cont = x[:, :self.cont_dim]
        x_cat  = x[:, self.cont_dim].long()
        ne = self.neigh_emb(x_cat)
        return torch.cat([x_cont, ne], dim=1)

    def get_embedding(self, x): return self.enc(self._unpack(x))
    def forward(self, x): return self.head(self.get_embedding(x))

print(f'Models defined: RoadMLP(8d) + TabularMLP({TAB_CONT_DIM}d+{NEIGH_EMB_DIM}d emb)')

Models defined: RoadMLP(8d) + TabularMLP(30d+4d emb)


## 10. 2-Branch CrossAttention Fusion

In [27]:
class TwoBranchFusion(nn.Module):
    def __init__(self, road, tabular, d_model=32, n_heads=4, drop=0.3):
        super().__init__()
        self.road, self.tabular = road, tabular
        self.proj_road = nn.Linear(road.emb_dim, d_model)
        self.proj_tab  = nn.Linear(tabular.emb_dim, d_model)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=drop, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.attn_ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(d_model * 2, d_model))
        self.ffn_norm = nn.LayerNorm(d_model)

        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, 16), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(16, 2))
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(32, NUM_CLASSES))

    def forward(self, xr, xb):
        er = self.proj_road(self.road.get_embedding(xr))
        eb = self.proj_tab(self.tabular.get_embedding(xb))
        seq = torch.stack([er, eb], dim=1)
        attn_out, _ = self.cross_attn(seq, seq, seq)
        seq = self.attn_norm(seq + attn_out)
        seq = self.ffn_norm(seq + self.attn_ffn(seq))
        er, eb = seq[:, 0], seq[:, 1]
        w = torch.softmax(self.gate(torch.cat([er, eb], dim=1)), dim=1)
        fused = w[:, 0:1] * er + w[:, 1:2] * eb
        return self.classifier(fused)

    def get_gate_weights(self, xr, xb):
        with torch.no_grad():
            er = self.proj_road(self.road.get_embedding(xr))
            eb = self.proj_tab(self.tabular.get_embedding(xb))
            seq = torch.stack([er, eb], dim=1)
            attn_out, _ = self.cross_attn(seq, seq, seq)
            seq = self.attn_norm(seq + attn_out)
            seq = self.ffn_norm(seq + self.attn_ffn(seq))
            er, eb = seq[:, 0], seq[:, 1]
            return torch.softmax(self.gate(torch.cat([er, eb], dim=1)), dim=1)

    def forward_ablation(self, xr, xb, disable=None):
        er = self.proj_road(self.road.get_embedding(xr))
        eb = self.proj_tab(self.tabular.get_embedding(xb))
        if disable == 'road': er = torch.zeros_like(er)
        elif disable == 'tabular': eb = torch.zeros_like(eb)
        seq = torch.stack([er, eb], dim=1)
        attn_out, _ = self.cross_attn(seq, seq, seq)
        seq = self.attn_norm(seq + attn_out)
        seq = self.ffn_norm(seq + self.attn_ffn(seq))
        er, eb = seq[:, 0], seq[:, 1]
        w = torch.softmax(self.gate(torch.cat([er, eb], dim=1)), dim=1)
        fused = w[:, 0:1] * er + w[:, 1:2] * eb
        return self.classifier(fused)

## 11. Dataset (2-input)

In [28]:
class FusionDS(Dataset):
    def __init__(self, xr, xb, y):
        self.xr, self.xb, self.y = xr, xb, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.xr[i], self.xb[i], self.y[i]

## 12. Training Utilities

In [29]:
def train_branch(model, X_tr, X_v, y_tr_, y_v_, epochs=60, bs=256, lr=2e-3, patience=12, verbose=True):
    model = model.to(DEVICE)
    tr_dl = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr_).long()), batch_size=bs, shuffle=True)
    v_dl  = DataLoader(TensorDataset(torch.tensor(X_v), torch.tensor(y_v_).long()), batch_size=bs)
    focal = FocalLoss(alpha=CW, gamma=2.0, label_smoothing=0.1)
    ordinal_loss = OrdinalPenalty(weight=0.3)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1, wait, best_sd = 0, 0, None
    for ep in range(1, epochs+1):
        model.train()
        for X, yy in tr_dl:
            X, yy = X.to(DEVICE), yy.to(DEVICE)
            logits = model(X)
            loss = focal(logits, yy) + ordinal_loss(logits, yy)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for X, yy in v_dl:
                preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
                labs.extend(yy.numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1: best_f1, wait, best_sd = f1, 0, {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: wait += 1
        if verbose and ep % 10 == 0: print(f'  Ep {ep:>3} val_f1={f1:.4f} best={best_f1:.4f}')
        if wait >= patience:
            if verbose: print(f'  Early stop ep {ep}')
            break
    model.load_state_dict(best_sd)
    return model, best_f1

def eval_test(model, X_te, y_te_, bs=256):
    model.eval()
    dl = DataLoader(TensorDataset(torch.tensor(X_te), torch.tensor(y_te_).long()), batch_size=bs)
    preds, labs = [], []
    with torch.no_grad():
        for X, yy in dl:
            preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
            labs.extend(yy.numpy())
    return f1_score(labs, preds, average='macro'), np.array(preds), np.array(labs)

## 13. Train Branches

In [30]:
print('\n=== Road-MLP (8d → 16d) ===')
road_model = RoadMLP(dim=Xr_tr.shape[1])
road_model, road_val = train_branch(road_model, Xr_tr, Xr_v, y_tr, y_v)
road_test, _, _ = eval_test(road_model, Xr_te, y_te)
print(f'  val={road_val:.4f} test={road_test:.4f}\n')

print(f'=== Tabular-MLP ({TAB_CONT_DIM}d+{NEIGH_EMB_DIM}d emb → 16d) ===')
tab_model = TabularMLP(cont_dim=TAB_CONT_DIM, n_neigh=N_NEIGH, neigh_emb_dim=NEIGH_EMB_DIM)
tab_model, tab_val = train_branch(tab_model, Xb_tr, Xb_v, y_tr, y_v)
tab_test, _, _ = eval_test(tab_model, Xb_te, y_te)
print(f'  val={tab_val:.4f} test={tab_test:.4f}\n')

print(f'{"Branch":<20} {"Val":>6} {"Test":>6}')
print('-'*34)
for n,v,t in [('Road-MLP (8d)',road_val,road_test),('Tabular-MLP (34d)',tab_val,tab_test)]:
    print(f'{n:<20} {v:>6.4f} {t:>6.4f}')


=== Road-MLP (8d → 16d) ===
  Ep  10 val_f1=0.3771 best=0.4122
  Early stop ep 15
  val=0.4122 test=0.4149

=== Tabular-MLP (30d+4d emb → 16d) ===
  Ep  10 val_f1=0.4961 best=0.5182
  Early stop ep 17
  val=0.5182 test=0.5216

Branch                  Val   Test
----------------------------------
Road-MLP (8d)        0.4122 0.4149
Tabular-MLP (34d)    0.5182 0.5216


## 14. 2-Branch Fusion Training

In [31]:
fusion = TwoBranchFusion(road_model, tab_model, d_model=32, n_heads=4).to(DEVICE)
print(f'\nFusion params: {sum(p.numel() for p in fusion.parameters()):,}')

tr_dl = DataLoader(FusionDS(torch.tensor(Xr_tr), torch.tensor(Xb_tr), torch.tensor(y_tr).long()), batch_size=256, shuffle=True)
v_dl  = DataLoader(FusionDS(torch.tensor(Xr_v), torch.tensor(Xb_v), torch.tensor(y_v).long()), batch_size=256)
te_dl = DataLoader(FusionDS(torch.tensor(Xr_te), torch.tensor(Xb_te), torch.tensor(y_te).long()), batch_size=256)

focal = FocalLoss(alpha=CW, gamma=2.0, label_smoothing=0.1)
ordinal = OrdinalPenalty(weight=0.3)

def eval_fusion(model, dl, disable=None):
    model.eval()
    preds, labs, all_probs = [], [], []
    with torch.no_grad():
        for xr, xb, yy in dl:
            xr, xb = xr.to(DEVICE), xb.to(DEVICE)
            logits = model.forward_ablation(xr, xb, disable) if disable else model(xr, xb)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            preds.extend(logits.argmax(1).cpu().numpy()); labs.extend(yy.numpy())
    return f1_score(labs, preds, average='macro'), np.array(preds), np.array(labs), np.vstack(all_probs)

# Phase 1: Warmup
print('\n--- Phase 1: Warmup (5 ep, branches frozen) ---')
for p in list(fusion.road.parameters()) + list(fusion.tabular.parameters()):
    p.requires_grad = False
opt = torch.optim.AdamW([p for p in fusion.parameters() if p.requires_grad], lr=2e-3, weight_decay=1e-4)
for ep in range(1, 6):
    fusion.train()
    for xr, xb, yy in tr_dl:
        xr, xb, yy = xr.to(DEVICE), xb.to(DEVICE), yy.to(DEVICE)
        logits = fusion(xr, xb)
        loss = focal(logits, yy) + ordinal(logits, yy)
        opt.zero_grad(); loss.backward(); opt.step()
    f1, _, _, _ = eval_fusion(fusion, v_dl)
    print(f'  Warmup {ep}/5 val_f1={f1:.4f}')

# Phase 2: End-to-end
print('\n--- Phase 2: End-to-end (60 ep) ---')
for p in fusion.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(fusion.parameters(), lr=2e-4, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)
best_f1, wait, best_sd = 0, 0, None
for ep in range(1, 61):
    fusion.train()
    for xr, xb, yy in tr_dl:
        xr, xb, yy = xr.to(DEVICE), xb.to(DEVICE), yy.to(DEVICE)
        logits = fusion(xr, xb)
        loss = focal(logits, yy) + ordinal(logits, yy)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(fusion.parameters(), 1.0); opt.step()
    sch.step()
    f1, _, _, _ = eval_fusion(fusion, v_dl)
    mk = ''
    if f1 > best_f1: best_f1, wait, best_sd = f1, 0, {k:v.cpu().clone() for k,v in fusion.state_dict().items()}; mk = ' *'
    else: wait += 1
    if ep % 5 == 0: print(f'  Ep {ep:>3} val_f1={f1:.4f} best={best_f1:.4f}{mk}')
    if wait >= 12: print(f'  Early stop ep {ep}'); break
fusion.load_state_dict(best_sd); fusion = fusion.to(DEVICE)

test_f1, preds, labs, test_probs_fusion = eval_fusion(fusion, te_dl)
print(f'\n{"="*60}')
print(f'2-BRANCH FUSION TEST MACRO F1: {test_f1:.4f}')
print(f'{"="*60}')
print(classification_report(labs, preds, target_names=CLASS_NAMES))

cm = confusion_matrix(labs, preds)
print('Confusion Matrix:')
header = ''.join(f'{"Pred "+n:>16}' for n in CLASS_NAMES)
print(f'{"":>16}{header}')
for i, name in enumerate(CLASS_NAMES):
    row = ''.join(f'{cm[i,j]:>16}' for j in range(NUM_CLASSES))
    print(f'{"True "+name:>16}{row}')

# Ablation
print(f'\n{"="*60}')
print('ABLATION STUDY')
print(f'{"="*60}')
results = {'Full model': test_f1}
for branch in ['road','tabular']:
    ab_f1, _, _, _ = eval_fusion(fusion, te_dl, disable=branch)
    results[f'w/o {branch}'] = ab_f1
print(f'\n{"Config":<20} {"F1":>6} {"Delta":>8}')
print('-'*36)
for k, v in results.items():
    d = f'{v - test_f1:+.4f}' if k != 'Full model' else '  base'
    print(f'{k:<20} {v:>6.4f} {d:>8}')

# Gate weights
print(f'\n{"="*60}')
print('GATE WEIGHT ANALYSIS')
print(f'{"="*60}')
all_w = []
with torch.no_grad():
    for xr, xb, yy in te_dl:
        all_w.append(fusion.get_gate_weights(xr.to(DEVICE), xb.to(DEVICE)).cpu().numpy())
all_w = np.vstack(all_w)
print(f'Mean gate weights:  Road={all_w[:,0].mean():.4f}  Tabular={all_w[:,1].mean():.4f}')
for cls, name in enumerate(CLASS_NAMES):
    mask_cls = labs == cls
    if mask_cls.sum() > 0:
        print(f'  {name}: R={all_w[mask_cls,0].mean():.3f} Tab={all_w[mask_cls,1].mean():.3f}')


Fusion params: 29,399

--- Phase 1: Warmup (5 ep, branches frozen) ---
  Warmup 1/5 val_f1=0.4934
  Warmup 2/5 val_f1=0.4894
  Warmup 3/5 val_f1=0.5009
  Warmup 4/5 val_f1=0.5126
  Warmup 5/5 val_f1=0.5196

--- Phase 2: End-to-end (60 ep) ---
  Ep   5 val_f1=0.5148 best=0.5181
  Ep  10 val_f1=0.5154 best=0.5181
  Early stop ep 13

2-BRANCH FUSION TEST MACRO F1: 0.5127
              precision    recall  f1-score   support

         Low       0.46      0.53      0.49       487
      Medium       0.56      0.43      0.49       885
        High       0.51      0.61      0.56       693

    accuracy                           0.51      2065
   macro avg       0.51      0.52      0.51      2065
weighted avg       0.52      0.51      0.51      2065

Confusion Matrix:
                        Pred Low     Pred Medium       Pred High
        True Low             256             141              90
     True Medium             190             381             314
       True High             106  

## 15. ★ v9: XGBoost + CatBoost ★

In [32]:
try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    from xgboost import XGBClassifier

# Flat features for trees (8 road + 31 tabular = 39)
X_flat_tr = np.concatenate([Xr_tr_orig, Xb_tr_orig], axis=1)
X_flat_v  = np.concatenate([Xr_v, Xb_v], axis=1)
X_flat_te = np.concatenate([Xr_te, Xb_te], axis=1)
NEIGH_COL_IDX = X_flat_tr.shape[1] - 1  # last col = neighbourhood index

print(f'\n--- XGBoost ({X_flat_tr.shape[1]} features, original data) ---')
xgb = XGBClassifier(n_estimators=500, max_depth=7, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                     eval_metric='mlogloss', num_class=NUM_CLASSES,
                     random_state=42, verbosity=0)
xgb.fit(X_flat_tr, y_tr_orig, eval_set=[(X_flat_v, y_v)], verbose=False)
xgb_preds = xgb.predict(X_flat_te)
xgb_f1 = f1_score(y_te, xgb_preds, average='macro')
print(f'XGBoost Test Macro F1: {xgb_f1:.4f}')
print(classification_report(y_te, xgb_preds, target_names=CLASS_NAMES))

# --- CatBoost ---
HAS_CATBOOST = True
try:
    from catboost import CatBoostClassifier
except ImportError:
    try:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'catboost'])
        from catboost import CatBoostClassifier
    except:
        HAS_CATBOOST = False
        print('CatBoost not available, will use 2-model ensemble')

if HAS_CATBOOST:
    # CatBoost needs categorical columns as int/str, not float.
    # Use DataFrame so neighbourhood column can have its own dtype.
    col_names = [f'f{i}' for i in range(X_flat_tr.shape[1])]
    neigh_col = col_names[NEIGH_COL_IDX]

    df_tr_cb = pd.DataFrame(X_flat_tr, columns=col_names)
    df_v_cb  = pd.DataFrame(X_flat_v,  columns=col_names)
    df_te_cb = pd.DataFrame(X_flat_te, columns=col_names)
    df_tr_cb[neigh_col] = df_tr_cb[neigh_col].round().astype(int).astype(str)
    df_v_cb[neigh_col]  = df_v_cb[neigh_col].round().astype(int).astype(str)
    df_te_cb[neigh_col] = df_te_cb[neigh_col].round().astype(int).astype(str)

    print(f'\n--- CatBoost ({X_flat_tr.shape[1]} features, neigh as categorical) ---')
    cb = CatBoostClassifier(
        iterations=500, depth=7, learning_rate=0.05,
        cat_features=[neigh_col],
        eval_metric='MultiClass',
        random_seed=42, verbose=100,
        early_stopping_rounds=50)
    cb.fit(df_tr_cb, y_tr_orig, eval_set=(df_v_cb, y_v))
    cb_preds = cb.predict(df_te_cb).flatten().astype(int)
    cb_f1 = f1_score(y_te, cb_preds, average='macro')
    print(f'CatBoost Test Macro F1: {cb_f1:.4f}')
    print(classification_report(y_te, cb_preds, target_names=CLASS_NAMES))


--- XGBoost (39 features, original data) ---
XGBoost Test Macro F1: 0.5117
              precision    recall  f1-score   support

         Low       0.57      0.39      0.47       487
      Medium       0.51      0.61      0.56       885
        High       0.53      0.50      0.51       693

    accuracy                           0.52      2065
   macro avg       0.53      0.50      0.51      2065
weighted avg       0.53      0.52      0.52      2065


--- CatBoost (39 features, neigh as categorical) ---
0:	learn: 1.0834609	test: 1.0845713	best: 1.0845713 (0)	total: 61.1ms	remaining: 30.5s
100:	learn: 0.8535130	test: 0.9227082	best: 0.9225780 (99)	total: 1.35s	remaining: 5.35s
200:	learn: 0.8066318	test: 0.9176779	best: 0.9175343 (189)	total: 2.64s	remaining: 3.93s
300:	learn: 0.7622976	test: 0.9169160	best: 0.9165866 (268)	total: 3.96s	remaining: 2.62s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9165866223
bestIteration = 268

Shrink model to first 269 iterati

## 16. ★ v9: Multi-Model Ensemble ★

In [33]:
print(f'\n{"="*60}')
print('ENSEMBLE')
print(f'{"="*60}')

_, _, _, val_probs_fusion = eval_fusion(fusion, v_dl)
val_probs_xgb = xgb.predict_proba(X_flat_v)
xgb_probs_te = xgb.predict_proba(X_flat_te)

if HAS_CATBOOST:
    val_probs_cb = cb.predict_proba(df_v_cb)
    cb_probs_te  = cb.predict_proba(df_te_cb)

    # 3-model weight search
    best_w, best_ef = None, 0
    for w1 in np.arange(0.05, 0.90, 0.05):
        for w2 in np.arange(0.05, 0.95 - w1, 0.05):
            w3 = round(1.0 - w1 - w2, 2)
            if w3 < 0.05: continue
            ens_p = w1 * val_probs_fusion + w2 * val_probs_xgb + w3 * val_probs_cb
            ef = f1_score(y_v, ens_p.argmax(1), average='macro')
            if ef > best_ef:
                best_w, best_ef = (w1, w2, w3), ef

    print(f'Best weights (val): Fusion={best_w[0]:.2f} XGB={best_w[1]:.2f} CatBoost={best_w[2]:.2f}')
    print(f'Val F1: {best_ef:.4f}')

    ens_probs_te = best_w[0] * test_probs_fusion + best_w[1] * xgb_probs_te + best_w[2] * cb_probs_te
    ens_preds = ens_probs_te.argmax(axis=1)
    ens_f1 = f1_score(y_te, ens_preds, average='macro')
    print(f'\n3-Model Ensemble Test Macro F1: {ens_f1:.4f}')
    print(classification_report(y_te, ens_preds, target_names=CLASS_NAMES))
else:
    # 2-model fallback
    best_ew, best_ef = 0, 0
    for w in np.arange(0.05, 0.95, 0.05):
        ens_p = w * val_probs_fusion + (1-w) * val_probs_xgb
        ef = f1_score(y_v, ens_p.argmax(1), average='macro')
        if ef > best_ef: best_ew, best_ef = w, ef
    best_w = (best_ew, 1-best_ew)
    print(f'Best weight (val): fusion_w={best_ew:.2f}')
    print(f'Val F1: {best_ef:.4f}')
    ens_probs_te = best_ew * test_probs_fusion + (1-best_ew) * xgb_probs_te
    ens_preds = ens_probs_te.argmax(axis=1)
    ens_f1 = f1_score(y_te, ens_preds, average='macro')
    print(f'\n2-Model Ensemble Test Macro F1: {ens_f1:.4f}')
    print(classification_report(y_te, ens_preds, target_names=CLASS_NAMES))


ENSEMBLE
Best weights (val): Fusion=0.30 XGB=0.10 CatBoost=0.60
Val F1: 0.5446

3-Model Ensemble Test Macro F1: 0.5207
              precision    recall  f1-score   support

         Low       0.57      0.41      0.47       487
      Medium       0.51      0.65      0.57       885
        High       0.56      0.48      0.51       693

    accuracy                           0.54      2065
   macro avg       0.55      0.51      0.52      2065
weighted avg       0.54      0.54      0.53      2065



## 17. Per-Class Threshold Optimization

In [34]:
print(f'\n{"="*60}')
print('PER-CLASS THRESHOLD OPTIMIZATION')
print(f'{"="*60}')

def apply_thresholds(probs, t):
    adjusted = probs.copy()
    for c, tc in enumerate(t):
        adjusted[:, c] -= tc
    return adjusted.argmax(1)

def neg_macro_f1(t, probs, y_true):
    return -f1_score(y_true, apply_thresholds(probs, t), average='macro')

if HAS_CATBOOST:
    ens_probs_v = best_w[0] * val_probs_fusion + best_w[1] * val_probs_xgb + best_w[2] * val_probs_cb
else:
    ens_probs_v = best_w[0] * val_probs_fusion + best_w[1] * val_probs_xgb

result = differential_evolution(
    neg_macro_f1, [(-0.5, 0.5)] * 3,
    args=(ens_probs_v, y_v), seed=42, maxiter=300, tol=1e-6)

print(f'Optimal thresholds: {[f"{t:.4f}" for t in result.x]}')
print(f'Val F1 with thresholds: {-result.fun:.4f} (vs {best_ef:.4f} without)')

tuned_preds = apply_thresholds(ens_probs_te, result.x)
tuned_f1 = f1_score(y_te, tuned_preds, average='macro')
print(f'\nThreshold-Tuned Test Macro F1: {tuned_f1:.4f}')
print(classification_report(y_te, tuned_preds, target_names=CLASS_NAMES))

print('Per-class F1 comparison:')
print(f'  {"Class":<10} {"Argmax":>8} {"Tuned":>8} {"Delta":>8}')
for c, name in enumerate(CLASS_NAMES):
    f_arg = f1_score((y_te==c).astype(int), (ens_preds==c).astype(int))
    f_tun = f1_score((y_te==c).astype(int), (tuned_preds==c).astype(int))
    print(f'  {name:<10} {f_arg:>8.4f} {f_tun:>8.4f} {f_tun-f_arg:>+8.4f}')


PER-CLASS THRESHOLD OPTIMIZATION
Optimal thresholds: ['-0.1803', '-0.1742', '-0.1765']
Val F1 with thresholds: 0.5470 (vs 0.5446 without)

Threshold-Tuned Test Macro F1: 0.5201
              precision    recall  f1-score   support

         Low       0.57      0.41      0.47       487
      Medium       0.51      0.64      0.57       885
        High       0.55      0.48      0.51       693

    accuracy                           0.53      2065
   macro avg       0.54      0.51      0.52      2065
weighted avg       0.54      0.53      0.53      2065

Per-class F1 comparison:
  Class        Argmax    Tuned    Delta
  Low          0.4737   0.4749  +0.0013
  Medium       0.5741   0.5707  -0.0034
  High         0.5143   0.5146  +0.0003


## 18. SHAP Analysis

In [35]:
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'shap'])
    import shap

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fusion.eval()
R_DIM = Xr_tr.shape[1]
B_DIM = Xb_tr.shape[1]

def model_predict_flat(flat_input):
    t = torch.FloatTensor(flat_input).to(DEVICE)
    xr = t[:, :R_DIM]
    xb = t[:, R_DIM:]
    with torch.no_grad():
        return torch.softmax(fusion(xr, xb), dim=1).cpu().numpy()

bg = np.hstack([Xr_tr[:100], Xb_tr[:100]])
test_flat = np.hstack([Xr_te[:200], Xb_te[:200]])

print('\nRunning SHAP KernelExplainer...')
explainer = shap.KernelExplainer(model_predict_flat, bg)
shap_values = explainer.shap_values(test_flat, nsamples=100)

road_names = ['length', 'su_Arterial', 'su_Collector', 'su_Residential',
              'elevation', 'slope', 'source', 'ROW_width']
tab_names = [
    'repair_count', 'segment_density', 'dist_to_traffic',
    'sl_risk_3', 'sl_high_3', 'sl_risk_7', 'sl_high_7', 'sl_med_7',
    'sl_risk_15', 'sl_high_15',
    'neigh_low_pct', 'neigh_med_pct', 'neigh_high_pct',
    'complaint_total', 'pothole_total', 'complaint_recent_3m', 'complaint_accel',
    'complaint_density', 'sl_complaints',
    'annual_precip', 'annual_snow', 'annual_freeze', 'annual_extreme',
    'annual_temp_range', 'prev_yr_precip', 'prev_yr_freeze',
    'precip_x_slope', 'freeze_x_street', 'snow_x_slope', 'temprange_x_street',
    'neighbourhood']
all_names = road_names + tab_names

if isinstance(shap_values, list):
    sv = shap_values[2]
else:
    sv = shap_values[:, :, 2] if shap_values.ndim == 3 else shap_values
mean_abs = np.abs(sv).mean(axis=0)

road_imp = mean_abs[:R_DIM].sum()
tab_imp = mean_abs[R_DIM:].sum()
total_imp = road_imp + tab_imp
print(f'\n{"="*55}')
print('SHAP Branch-Level Importance (High-risk class)')
print(f'{"="*55}')
print(f'  Road-MLP:     {road_imp/total_imp*100:.1f}%')
print(f'  Tabular-MLP:  {tab_imp/total_imp*100:.1f}%')

top_idx = np.argsort(mean_abs)[::-1][:15]
print(f'\nTop 15 features by mean |SHAP|:')
for i, idx in enumerate(top_idx):
    idx = int(idx)
    name = all_names[idx] if idx < len(all_names) else f'feature_{idx}'
    print(f'  {i+1:>2}. {name:<30} {mean_abs[idx]:.4f}')

fig, ax = plt.subplots(figsize=(10, 6))
top_names_plot = [all_names[int(i)] if int(i) < len(all_names) else f'f_{i}' for i in top_idx[:15]]
top_vals = [mean_abs[int(i)] for i in top_idx[:15]]
ax.barh(range(15), top_vals[::-1])
ax.set_yticks(range(15)); ax.set_yticklabels(top_names_plot[::-1])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('CityBrain v9 — Top 15 Feature Importance (High Risk)')
plt.tight_layout(); plt.savefig('shap_importance_v9.png', dpi=150); plt.show()
print('Saved shap_importance_v9.png')


Running SHAP KernelExplainer...


  0%|          | 0/200 [00:00<?, ?it/s]


SHAP Branch-Level Importance (High-risk class)
  Road-MLP:     7.0%
  Tabular-MLP:  93.0%

Top 15 features by mean |SHAP|:
   1. sl_risk_3                      0.0232
   2. sl_risk_15                     0.0222
   3. sl_risk_7                      0.0188
   4. sl_high_7                      0.0147
   5. sl_high_3                      0.0123
   6. pothole_total                  0.0052
   7. neigh_high_pct                 0.0050
   8. annual_snow                    0.0045
   9. annual_temp_range              0.0034
  10. annual_extreme                 0.0033
  11. length                         0.0033
  12. sl_high_15                     0.0031
  13. neighbourhood                  0.0029
  14. annual_freeze                  0.0028
  15. annual_precip                  0.0028
Saved shap_importance_v9.png


## 19. FINAL COMPARISON

In [36]:
all_results = [ens_f1, tuned_f1]
best_name = 'Threshold-Tuned Ensemble' if tuned_f1 >= ens_f1 else 'Ensemble'
best_test = max(all_results)

print(f'\n{"="*60}')
print('FINAL COMPARISON — ALL VERSIONS')
print(f'{"="*60}')
print(f'{"Model":<55} {"Test Macro F1":>12}')
print('-'*69)
print(f'{"v1 Baseline (old 3-class, concat fusion)":<55} {"0.3915":>12}')
print(f'{"v2 Enhanced (old 3-class)":<55} {"0.4358":>12}')
print(f'{"v5 GatedFusion (rebalanced)":<55} {"0.4241":>12}')
print(f'{"v6 CrossAttention Fusion":<55} {"0.4400":>12}')
print(f'{"v6 Simple Ensemble (Fusion+XGB)":<55} {"0.5009":>12}')
print(f'{"v7 3-Branch Fusion":<55} {"0.5145":>12}')
print(f'{"v7 2-Model Ensemble (Fusion+XGB)":<55} {"0.5209":>12}')
print(f'{"v8 Threshold-Tuned Ensemble":<55} {"0.5178":>12}')
print(f'---')
print(f'{"v9 Road-MLP (8d, physical only)":<55} {road_test:>12.4f}')
print(f'{"v9 Tabular-MLP (34d, context+neigh emb)":<55} {tab_test:>12.4f}')
print(f'{"v9 2-Branch Fusion (d_model=32)":<55} {test_f1:>12.4f}')
print(f'{"v9 XGBoost":<55} {xgb_f1:>12.4f}')
if HAS_CATBOOST:
    print(f'{"v9 CatBoost (native categorical)":<55} {cb_f1:>12.4f}')
    n_models = 3
else:
    n_models = 2
print(f'{"v9 " + str(n_models) + "-Model Ensemble":<55} {ens_f1:>12.4f}')
print(f'{"★ v9 Threshold-Tuned Ensemble":<55} {tuned_f1:>12.4f}')
print(f'\n{"="*60}')
print(f'BEST: {best_name} = {best_test:.4f}')
print(f'Improvement over v6 (0.5009): {best_test - 0.5009:+.4f}')
print(f'Improvement over v7 (0.5209): {best_test - 0.5209:+.4f}')
print(f'Improvement over v8 (0.5178): {best_test - 0.5178:+.4f}')
print(f'{"="*60}')

print(f'\nAll {len(df):,} samples used.')
print(f'v9 key changes:')
print(f'  1. CLEAN feature separation: Road=physical(8d), Tabular=context(34d)')
print(f'  2. Neighbourhood entity embedding ({N_NEIGH}→{NEIGH_EMB_DIM}d, was {N_NEIGH}d one-hot)')
print(f'  3. Multi-scale spatial lag (3/7/15-NN)')
print(f'  4. Complaint density (complaints / road length)')
print(f'  5. CatBoost with native categorical support')
print(f'  6. Feature overlap between branches: ZERO')


FINAL COMPARISON — ALL VERSIONS
Model                                                   Test Macro F1
---------------------------------------------------------------------
v1 Baseline (old 3-class, concat fusion)                      0.3915
v2 Enhanced (old 3-class)                                     0.4358
v5 GatedFusion (rebalanced)                                   0.4241
v6 CrossAttention Fusion                                      0.4400
v6 Simple Ensemble (Fusion+XGB)                               0.5009
v7 3-Branch Fusion                                            0.5145
v7 2-Model Ensemble (Fusion+XGB)                              0.5209
v8 Threshold-Tuned Ensemble                                   0.5178
---
v9 Road-MLP (8d, physical only)                               0.4149
v9 Tabular-MLP (34d, context+neigh emb)                       0.5216
v9 2-Branch Fusion (d_model=32)                               0.5127
v9 XGBoost                                                    0.